# Turnkey: debug a run
Connect one paired case to the runtime events that explain what happened.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path
import yaml

workspace = Path(tempfile.mkdtemp(prefix='turnkey-debug-'))
config = yaml.safe_load(Path('configs/runs/smoke.yaml').read_text())
config['run']['out_dir'] = str(workspace / 'outputs')
config_path = workspace / 'smoke.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
run_result = subprocess.run([sys.executable, '-m', 'turnkey.cli', 'run', '--config', str(config_path)], check=True, capture_output=True, text=True)
run_dir = Path(run_result.stdout.strip().splitlines()[-1])
cases = [json.loads(line) for line in (run_dir / 'cases.jsonl').read_text().splitlines()]
events = [json.loads(line) for line in (run_dir / 'events.jsonl').read_text().splitlines()]
len(cases), len(events)

In [ ]:
case_id = cases[0]['case_id']
view = subprocess.run([sys.executable, '-m', 'turnkey.cli', 'inspect', str(run_dir), '--case-id', case_id, '--json'], check=True, capture_output=True, text=True)
detail = json.loads(view.stdout)
case_events = [event for event in events if event.get('case_id') == case_id]
assert detail
{'case_id': case_id, 'event_kinds': sorted({event['kind'] for event in case_events})}